<a href="https://colab.research.google.com/github/Arif118/Weather-Prediction/blob/main/Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests as r
import pandas as pd
import random
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

In [2]:
API_KEY = '0dcff6d1048e25ef4de7711889a9917e'
weather_data = []

In [3]:
def get_weather_data(data):

    try:
        country = data['sys']['country']

    except KeyError:
        country = 'Unknown'

    features = {

        'Temp': data['main']['temp'] - 273.15,

        'Pressure': data['main']['pressure'],

        'Humidity': data['main']['humidity'],

        'Wind_speed': data['wind']['speed'],

        'Cloudiness': data['clouds']['all'],

        'Country': country,

        'Target': data['weather'][0]['main']
    }

    weather_data.append(features)

In [4]:
for i in range(20):

    lat = random.uniform(-60, 60)

    lon = random.uniform(-100, 100)

    url = (
        f"https://api.openweathermap.org/data/2.5/weather?"
        f"lat={lat}&lon={lon}&appid={API_KEY}"
    )

    response = r.get(url)

    data = response.json()

    get_weather_data(data)

In [5]:
df = pd.DataFrame(weather_data)
df.head()

,Temp,Pressure,Humidity,Wind_speed,Cloudiness,Country,Target
0,6.46,1017,94,14.14,100,Unknown,Clouds
1,26.43,1010,76,1.51,100,CD,Clouds
2,27.85,1008,73,0.77,100,Unknown,Clouds
3,25.87,1014,76,7.04,100,Unknown,Clouds
4,42.78,1006,5,5.95,0,CM,Clear


In [6]:
X = df.drop('Target', axis=1)
y = df['Target']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
numeric_features = [
    'Temp',
    'Pressure',
    'Humidity',
    'Wind_speed',
    'Cloudiness'
]

categorical_features = [
    'Country'
]

preprocessor = ColumnTransformer(
    transformers=[

        (
            'num',
            StandardScaler(),
            numeric_features
        ),

        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        )
    ]
)

In [11]:
pipe = Pipeline([

    ('preprocessor', preprocessor),

    (
        'model',

        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )
    )
])

In [12]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Temp', 'Pressure',
                                                   'Humidity', 'Wind_speed',
                                                   'Cloudiness']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country'])])),
                ('model', RandomForestClassifier(random_state=42))])

In [14]:
y_pred = pipe.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

Accuracy : 0.5000

Classification Report:

              precision    recall  f1-score   support

       Clear       0.00      0.00      0.00         1
      Clouds       0.50      1.00      0.67         2
        Rain       0.00      0.00      0.00         1

    accuracy                           0.50         4
   macro avg       0.17      0.33      0.22         4
weighted avg       0.25      0.50      0.33         4



In [15]:
input_data = pd.DataFrame([{

    'Temp': float(input('Temperature: ')),

    'Pressure': float(input('Pressure: ')),

    'Humidity': float(input('Humidity: ')),

    'Wind_speed': float(input('Wind Speed: ')),

    'Cloudiness': float(input('Cloudiness: ')),

    'Country': input('Country: ')
}])

Temperature: 37
Pressure: 10121
Humidity: 97
Wind Speed: 76
Cloudiness: 50
Country: PK


In [16]:
prediction = pipe.predict(input_data)

print("Predicted Weather Condition:")

print(prediction[0])

Predicted Weather Condition:
Clouds
